# Stardox Email Scraper

Takes a list of GitHub usernames and scrapes their public email addresses using a headless browser.

**How it works:**
1. Spins up headless Chromium via Playwright (downloads its own browser — no system Chrome needed)
2. Visits each user's GitHub profile
3. Looks for email in the profile sidebar (JS-rendered)
4. If no email on profile, checks their commit history (.patch files)
5. Outputs username:email pairs as a downloadable CSV

In [ ]:
# Install Playwright + its own bundled Chromium (does NOT use system Chrome)
!pip install -q playwright nest_asyncio pandas tqdm
!playwright install chromium
!playwright install-deps chromium

In [ ]:
import re
import asyncio
import nest_asyncio
import pandas as pd
from tqdm.notebook import tqdm
from playwright.async_api import async_playwright

nest_asyncio.apply()

EMAIL_RE = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')
IGNORE_PATTERNS = ['noreply', 'users.noreply.github.com', 'github.com', 'githubusercontent']


def is_valid_email(email):
    if not email:
        return False
    email_lower = email.lower()
    for pattern in IGNORE_PATTERNS:
        if pattern in email_lower:
            return False
    return True


async def start_browser(github_cookie=None):
    """Launch headless Chromium via Playwright async API."""
    pw = await async_playwright().start()
    browser = await pw.chromium.launch(headless=True)

    if github_cookie:
        context = await browser.new_context()
        await context.add_cookies([{
            'name': 'user_session',
            'value': github_cookie,
            'domain': '.github.com',
            'path': '/',
            'secure': True,
        }])
        page = await context.new_page()
        print('Browser started with GitHub session!')
    else:
        page = await browser.new_page()
        print('Browser started (anonymous — profile emails will be hidden)')

    return pw, browser, page


async def stop_browser(pw, browser):
    """Clean up browser and playwright."""
    try:
        await browser.close()
    except Exception:
        pass
    try:
        await pw.stop()
    except Exception:
        pass


async def scrape_email_from_profile(page, username):
    """Visit GitHub profile and extract email from page text."""
    try:
        await page.goto(f'https://github.com/{username}', wait_until='networkidle', timeout=20000)
        await page.wait_for_timeout(2000)

        body_text = await page.inner_text('body')

        emails = EMAIL_RE.findall(body_text)
        for email in emails:
            if is_valid_email(email):
                return email

        source = await page.content()
        mailto_matches = re.findall(r'mailto:([a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,})', source)
        for email in mailto_matches:
            if is_valid_email(email):
                return email

    except Exception:
        pass

    return None


async def scrape_email_from_commits(page, username):
    """Get email from user's commit .patch files."""
    try:
        await page.goto(f'https://github.com/{username}?tab=repositories&type=source',
                         wait_until='networkidle', timeout=20000)
        await page.wait_for_timeout(1000)

        repo_elements = await page.query_selector_all('a[itemprop="name codeRepository"]')
        repo_names = []
        for el in repo_elements[:3]:
            name = await el.inner_text()
            repo_names.append(name.strip())

        if not repo_names:
            return None

        for repo_name in repo_names:
            try:
                await page.goto(
                    f'https://github.com/{username}/{repo_name}/commits?author={username}',
                    wait_until='networkidle', timeout=20000)
                await page.wait_for_timeout(1000)

                commit_links = await page.query_selector_all(
                    f'a[href*="/{username}/{repo_name}/commit/"]')

                for commit_link in commit_links[:5]:
                    href = await commit_link.get_attribute('href')
                    if not href or '/commit/' not in href:
                        continue

                    label = (await commit_link.get_attribute('aria-label')) or ''
                    text = (await commit_link.inner_text()) or ''
                    if 'merge' in label.lower() or 'merge' in text.lower():
                        continue

                    if href.startswith('/'):
                        href = 'https://github.com' + href

                    await page.goto(href + '.patch', timeout=15000)
                    await page.wait_for_timeout(1000)

                    page_text = await page.content()

                    from_match = re.search(r'From:.*?<([^>]+@[^>]+)>', page_text)
                    if from_match:
                        email = from_match.group(1)
                        if is_valid_email(email):
                            return email

                    emails = EMAIL_RE.findall(page_text[:3000])
                    for email in emails:
                        if is_valid_email(email):
                            return email

            except Exception:
                continue

    except Exception:
        pass

    return None


async def scrape_email(page, username):
    """Try profile first, then commits."""
    email = await scrape_email_from_profile(page, username)
    if email:
        return email
    return await scrape_email_from_commits(page, username)


print('Functions loaded. Ready to scrape.')

In [ ]:
# ===========================================
# GITHUB SESSION COOKIE (required to see profile emails)
#
# How to get it:
# 1. Log into github.com in your browser
# 2. Open DevTools (F12) -> Application -> Cookies -> github.com
# 3. Find the "user_session" cookie and copy its value
# ===========================================

GITHUB_COOKIE = ""  # paste your user_session cookie value here

# ===========================================
# PASTE YOUR USERNAMES BELOW (one per line)
# ===========================================

usernames_input = """
xoxoyama69-bot
jannisg
mixcoooo
dinuka-rp
koa-rod
icarusoars
ddiegosr
jointkaru-source
MMitsuha
Dryja
mergho28
HarKro753
turbobeest
kkoggabim-cpu
turumtaev
numotion
marek-lar
Iskrata
kiramyby
mopdoc
raysubham
MilesHan
jaeyoung-l
david-botelho-mariano
wstdxmtns
andersonluizaxion
longwang-me
yuyuzhang
hexian2001
workingaichau-dot
Fulisike
lovanvo
Mr-KID-github
Andres620
tianpan
h2zi
karanjilka
Edd-G
emanuelediluzio
Delavari-Alireza
HYKQL-K
ishii2025buziness
ltaragi
AntiEris
kungfubaozi
VelvetAbyss
GravityZenAI
Franzferdinan51
dengdai68
boris-lok-pentadoc
patricka3125
vibes666888
princesingh-ai
juyongSong
MurfWorks
wilbeibi
jchen42703
Th3Sp3ct3R
sco09
syedamaann
devnazarchuk
ali-fidian
cgarciga
zibo2025
JoJossd
angelafeliciaa
neyham
itechdom
fivemeepo
bukanyrrenek
startagain2016
kellanlab
nilya
zervin
oxedom
mathewjustin
shonve
olyakashs
1SuperEmployee
joseluis2g
q3874758
g0TTAnakedTHeM
travellerwjoe
xdJumpING
mhz-tamb
SeOgi-Tsu
idoo
mhd-nour-khalifa
vlandyao
ishan-nitj
cconvalexius
abhijitthorat999-byte
serg-alexv
moalaoudi
bolohori
notanios
kylezk777
matthieuh
mourginakis
Lostendalfah1980
manoffrfccff
AB6806
FungXF
SombreroPatrick
evgenyus
DewmikeAmarasinghe
tianyicui
dyammarcano
chibicode
vmouradev
willow15
dimkk
lighthousemacro
leroyclarkejr
netfishx
notguerz
ppazos
romain-girardi-eng
heydemoura
brunobarros
dmitrii-khizbullin
pvelleleth
tony0x10
barrycarrjr
babooppa666
Rafewey
yangshangwei
Nostalogicwh
tier2tech-tian
schinnam
qsbrlgl
zulu-caPWN
reeves-48777
QuiteDiam0nd
CyberSys
middas-0706
JenjiLee
p-aliberkay
nksrentas
Traderfuz
radugrecu97
wendelstam
Jioyzen
regreg0706
duajay
danialbka
cadetmaze
Nekoiii
cryptoddness
Rovenwu777
wjn6
897578
burakroket
hyuni0316
wohckcin
JorgeMoreira-com
prdwyer
Havorld
Yinghai75
alexfrih
sudoMktemp
SilverHairedElfBabe
hpcnt-dan
andcortez
digitsisyph
khanhld1910
mareox
sgr0691
ari-real
stackconsult
dominikandrewtichy
zbulatt
imranbarbhuiya
legendhimself
niallam22
aisyser
hmarques98
troydildine
guhenriq
defly
zhulianyu
ruochenchenchen
rivadmorin
mahsumaktas
adamNewell
jipsy123
JulioDinizN
adnahmed
pawneshwar
huachuanchuan
Lemon4044
goncaloalves
yujiahaol68
tehtbl
zye3776
fattoh
c4kar
pablodiazv
TZzzzzzzzzzzzzz
lzpDark
dogterbox
darshangoswami
taviks
vibeyclaw
Tetetetetetetet
harishkanna-019
piyush123
llhhtt7788
WaterGGAI
xirui-li
Momoyu404
liyanghbar
digitarald
luist18
AtotheY
Perefin
AlankritVerma01
davidgomes
VincentiusJacob
proulxdev
nclsndr
mfekadu
sunilnanjiani
bmetcalf21
jestersimpps
travistang1910
aks861999
tedstonne
remoale
GenocideStomper
fedorn
RockieLiu
Neldarklix
pptrace
atheeq-rhxn
deepakkumarnd
yanmhlv
ketvector
Dorilitre
isalicema
Kenny-bi
afreedi-lp
aeromech-1
loic9654
vladalive
migosJava73
j-how
luizadevops
Jameswanttolearn
makeavish
Ali-Meddioui
famadeuser
mitixx
tangbing-xm
tvds04
MarsYoung
brittikbasu
ZhaoDezhang9
zaohello
Tonnodoubt
lawlort32
FeiLiuEM
yuxiuzhiai
shaybizking
gilsedition
rs0125
ranareinsit
jamesmilkybond
ron-fdx
lonsov
3588
mushroomlb
wildcherry500
takijaki516
fqwtt
jianfenghuo
marco-marcox
li77724121
jaseemts
commitToStahl
jboner
KayvanKarim
ZohaibAhmad-otifsolutions
ntdrun
vipbgl123
guthyerrz
PauloPhagula
tangshengsong
guoshivi
bartigorfs
Alphav00
guyramone
flosommerfeld
ksy-fish
adamscybot
Gxmestk
stealthy02
viokuma
rovanchung
noahkiss
wangrocky
Jack-PKU
merrygreek
Sriram-Merugu
Mertbaturg
0xtkey256
DMC-DDPS
hau823823
Shaubie
joobn72
xhoxye
qwamicodes
HarjjotSinghh
pfries
ncrown
youssef358
ofurkanuygur
ponderingBGI
andrewfurman
philipshurpik
kapamelia
Spark-chenlin
LoneMadTuna
cahangeorge
MehinZ
Garcke
FanWaterWater
b1gchoi
jkramer5103
protream
tianjicy
McGH7
a454511125-dotcom
hhtttyy
SandwormAI
EE-Azura
sukram-ai-dev
thinkace
sukilll
EarthXP
mp2123
073palmer
AnkittChauhan
us2393
Apoorvgarg-creator
22mithil
2799662352
jinjiaKarl
wxnacy
wr-chen666
syedrizwansrs
0xstackforge
athallabf
nathaniel-yuri
gauched
Divide-By-0
akhbar0w0
kaikauux
cainky
user74674
StannyByte
Xuyiyang23333
cy20lin
happydl
Gab-Angel
Nass-boop
agi-2026
jomojomo878
mad2p
luciusrockwing
jafarscharm
hudada-hub
Markjinli
dkleptsov
diegogit03
jentix
sayakuro
Nsy3a
HallWayLab
gzchen008
zktzktzkt
lyydsheep
YigWoo
cooladam201212
Xiangyu2141480
dlxmax
iag520
Player-191
JamesYoungZZ
qq36545
Aarya-Banthia
yalepi966-png
giorgio-zamparelli
PANDAcodeur
kellerman0
Wenkun2001
schemafault
koistya
raymondlei90s
codewithiulian
kayznari
kaisermoon
DIOovo
Kaijun-Luo
aoulaa
hsetially
sihaogu32
mylkrex
hifar
marcelooblan2016
fpatseas
laxmanclo
Masa-Kawa
bi-nguyen
weiched
Ridhvik-2024
wgsjack199213
0xanrelins
liy-cn
nicovalencia
39Er
Juchonghao
isaipanelinha
for2cold
hoangquocvietuet
chentao4183
loqimean
dineshndb
NailoEspectral
talha1503
itsmeminsithu
jonahinthewhale
ArnoFrost
upgraide
lot3sgh
wiredvibez
rafaelmatsuyama
XiaYiHann
ddwinhzy
dsdashun
miladvlp
yuush10
zsgvivo
aristoler
yinmazhong
AnttiKesti
CalvinSturm
banshanshenlou
sealala
an-lee
Chihong-Ng
bravemata
Partridge12
cbt-developers
chankingAI
jannolan200211-ship-it
rokokorn
kotaroyamazaki
443495431
GISER-KING
PokIsemaine
ttiimmothy
linwuqu
sanli036
Wsrgod
A2233479373
77777R7
WenwenXdaddy
leonjames-404
hei6775
YAORUNY
gostop54
Lauredmarin
JacksonHe04
stefeniefyjb
AnsImran
mssnzxm
DanielZ-28
praneeth9
alayna-devs
antmilk537
trry-hub
techinterns-ecloset
RDuke907
WangYI-JIE
Tevic
Twoone94
wangrunlin
mastiico
model-plus
Anionex
Radicalpro
airskywalker466-ui
Ainmymind
zhongdeming428
paulj-cc
lukeanthony007
straw-m
warren618
lazyGao-bit
shtoni
RiceParty
hd19820806
dziunincode69
wqbill
nailcagri
YanLin-Quinne
vedi108
runawaydevil
youpaimmian
ferranrod
Chopin132
hellooh1989
nayan-mehta
JackyChen2013
Guillaumedrlz
daakew
Shy-Plus
MattJan
Sasaki-kojiroo
Niremizov
DeightonLLM
joda49
Dostoevskey
lanyasheng
Citadel404
aayushkrm
malanjp
corvo007
ClareCoderAG
rawhit-r
Ji-Yuhang
zhiyozhao
ericzzhou
eric-yami
tasksolver
ARKL5
huang-baixin
syahh-coder
CyiHulsta
wen-haoming
elgca
imllz
IBeyondy
pxc1833
S4GU4R0
russellwmy
CommanderXL
niragireclaudette23-boop
kukenei
roy4222
x-zheng16
iKlare914
chsit
FanqingM
sunpenglv
ANyoneCC
timeju
xxgzzp
PalmPalm7
BlancoBAM
serenditipy-AC
Saraph1nes
2yyhsy2
wilhelmjung
hhhey-lw
JiamingChen1234
Hemanthk1099
InCoB
glowdan
chromaphase75
srijanshukla18
jimezsa
kalabro
bluesky2019
TD9999abc
nyflyer
Rick3129
MaksuCode
dantecamoz11cell
Noatitauaggi
zouhuigang
Luxebug
j14v
EternalRights
Tourist1864
nasle
mypisme
cnekol
kun0523
ShousenZHANG
BoozeLee
liusongxiang
aakaka525-design
Mebigger
xlboy
PeixinLu
joke-lx
qiaohaoforever
charilaouc
hueyexe
andrewparkk1
fire17
airfoxfull
gochiel
urwithajit9
jonkyu
lightstrike
kainwu
JonathanF07
SuperLucas
brucewang5638
XiejiLi
wzl71348-create
Reuben1987AI
kargozerov
R3LAMP4GO
L33chKing
bamboovir
recuriax
DarylFernandes99
Ntrakiyski
Nawfay
asaficontact
jeremygao2020
ameen-saeed
corvid-agent
jiuhuiyi
exploit1st
MuadDib10193
seeARMS
SandroWeber02
307537855
jorben
thakorneyp11
kelseyrae
3233509723
AlwaysLoveme
15cm
User00092
fangqiluxatu
xcc9526
yingyuankai
WangYuBo
Chyrain
ljh2023
ElTimuro
BQXBQX
91xinwei
Pytorchlover
secflag
pao-ying
1parado
ziphell
Anacoder1
mintseok
HealerKnight
johnzhao0520
gaurav-jo1
bbuugg
PassionetReve
alanhou-0220
cyl-jsl
ZeLv01
d0ublecl1ck
LCCapAdvisors
raosirui
f1basik
wu0x1d19
shootermusic
GargantuaX
ta93abe
leyunpeng75
Huchangzhi
jink-e
xingorg1
Ratzielor
MasterShadow097
hippohua
liuxiaogang
terminatoriust800-openclaw
zjhr
liuming-dev
panda-12138
zhangyibin1976
jerldev
HongJinjin9
patciello
Asgerov-Elnar
lbrealdev
whatsonut
wazder
KeisukeYamashita
taramakage
JonahFSD
icommitfelones
d0nda
Sakuralaaa
FraArchi
SlavaR07
stbdang
yushuinanrong
zabuxx
Osami203
anieve01
lokesh-danu
vuon9
noisyblue
zzxxvvzz
jimdx
piyush230502
PennyFlipperBot
avatar-lavventura
vvovv47
Men6d656e
harishpiyer
theryanpereira
moyan123456
benbenwu1
Hux-Z
amined159
kugruw
Nickwang684
junieberry
DeltaFarce
jk831124yuan
VladimirShleyev
mrhoric
nanmuze
breadbread1984
xysDavid
Yuis1
AlihanSDev
john8tom
ZhangCheng-zh
Fei-Good
stormgbs
massolala
skokinkoba
kejunxiao
NewMaxx76
Sirius-chen
nxm77
372572571
baedalus
ManCam87
wajdiJomaa
GagandeepLubana
pkwzsqsdly
charsea
AllA965
alobato
ecmoment
tryingtobuildthings
dengzhile3-create
alidinc33
gaozhenxiang469-cyber
MiloradLalovic
krpritam
grafgooseman
dev-gboy
Suryanshu-Nabheet
jaydiamond42
chenwanqq
sahirjamal
krishnaTORQUE
Riedell
UtchiKKotori
abhiiiman
gconsigli
ih-zonaid
Archger
derinbarutcu17
biernacki
muathejamil
justin-mc-lai
yzym1119
ch1kim0n1
zhouzhuojie
ht08088818i
wtypty2
huhuhuhr-1
lld98523
zj1123581321
rookiegc0
N0laa
marcelfolaron
akavictor2006-dotcom
sangyoz
igavinshang
linism
JonCSGuy
kfwalder
dev-olly
osintual
uneuro
kucukkanat
sebastianjlopez
badplay
Suyw-0123
Giovix0
danialzivehdar1992-hue
light-and-caffe-latte
anslt
citadelaqua
krishx64
posrix
abronte
alan199501
hwb96
CkeWMX
LiCaipu
liicheng
brainchen2020
brightwgm
vimalinx
tansunyj
evilnull
iamrahulroy
imprrf
iamhungry
MoKhajavi75
rexliu0715
andregaio
robchapman
El-anqiao
Alan-512
anh3taynguyen
CarolineCheng233
nuxnu
min6choi
LaiZhou
ahanover2024
pzhifeng
chaoslife
ivancgz
Miller547719886
baboonwu
rilnermucio
inwaylaw
fantiny
ajnahavandi-dev
marcosdayanm
nyudenkov
zhangjiluo-com
LocNgoXuan23
wpyAI
lifujie1992-wq
qiangu
ma2ong
MarcusXiaoFan
AlwaysKO
harrisonpy
ZKLOCK
eidos11
serima
shantmarouti
PratikPaudel
mvicari
ax128
TCadet
dpk-a7
karlmfvillanueva
mstump
zhangchier
fire9
SivaramPg
RuyXu
williambedard
vshen009
2shou
arugami-archive
cesarnieto
wgwdszerg
zmier
spyrae
jeshwan
w2049w
codedev-xy
Ralphbaer
Akshitakeshav
agzis
spookyuser
Timor68
oventh
sunspearless
lubomirblazekcz
Mr-Swapnil25
anton-karlovskiy
PsyGik
chance9077
changyongyong
water7th
coollela
ChiChasesCheese
yankemeng2-droid
benyl-nie-2
mikuh
1eo1ui
ald2004
CaoZhen
bing-deng
haishanghuafan
paomoshuilingling
s045pd
xnxy
sultanfarizbythen
wf85878703
lilingqian
mileson
naokeziteng
hjpotter1
lihuanshuai
wjlixianling
B1ithe
theifoxxxxxxxx-boop
aworki
Drmiaomiao
panxiaopan-ai
zhangjiefengd
vimbackground
max1874
user-NmYh
liaotuo
feomao
farazah-20
rapina
highwaydre
travensolli
handsomeko
TahaGHZ
LuaanNguyen
Xyzjesus
chenyjade
dmrbsio
Brunwo
boredcoderyt
GuilhermeSK2
Kirill89
travelhawk
ChrisKe89
v1lio
milos018
xyuer
msichterman
twt1978
razr001
theLucius7
zhuqling
gaoguosheng
nannanxueAI
dmechea
135e2
moise5andrad3
milicevicante
zhangfisher
gorgolo
Rolloniel
AhmedA-afk
baland7777777
VainEffort
FranzLy
muhifni
testejb
braxtonROSE4
lynch0571
wangruo91
geekmanvip
jcb850
wzm-ilovecplusplus
wareli
bazinga012
youa01093-sketch
elainehello
sleepers-xu
Ckxkx
lzj960515
hughgramel
SilenceNobike
charliedev00-ctrl
sergeylzc
MiaoDX
Y7NINE
xyh0693
jobsteven
keyingfu0
kumarbisen
Kebabist
RoloVoid
ahmetakyapi
chikobara
super-atom
alfianridwan
5unnyWind
Zscozer
laikee99
ganzizi
thehouseholdofshz
MuXsi
chenyuanqi
chifenglovekele
coolwcan
javie007
zwidny
HaiBoWon
acta0724
baracchande
WDesperadoK
ilkerozgedik
kacigaya
emre-turan
Raboo
dutchiegtb
drew458
xD-saleem
MaGonglei
xCasparx
ACGpp
fenxw
TIANYI005
transaurus
zhouxihong1
jpeirson
StoyKK
liuziheng20091106
tophgg
A1pha3
cyber-zengzhiyong
fuxuemingzhu
scostello
eladchen
zoroisnice
Frank1016
igoroctaviano
0x12F5CA9
yuyi1
rainbowcatcher
osmandkitay
xufeiyu8
dbxndn
HemantKumar01
georgejin110
zx9597446
wingfordream
tianlin
songlin51
ccsvip
jasonatcb
mynoveldownloads
ibey0nd
Rainyrou
yibaoshan
StarringJoe
kyj0503
t-fujise-gbc
Celivincy
aporpa
Likenttt
strauss-vasconcelos
thiemonipro
worldlightyjx
Fradeet
akaponym-byte
ruoyiya
robinfang
twidawski
wufulin
jeremysong88
bohdan147
aiyeming2025-glitch
wulinunu
Jolanr10
ppoloskov
hugobaum
saychuwho
dodolalorc
Cr1t-GYM
weilaikeqi789
xiaolu473
pengchengu
rafaelkamimura
databasegub
CharleyXu
CxMYu
sup-lh
xudean
BarryAllan76
zc0982
yechen0704
Hyao111
CXRui
zlulan
YourSigh
Meng20250915
antoniel
DiegoSanchez16
XYTIN
mihohoi0322
FantDing
vrazdalovschi
akmalovaa
sUfresd
LiamVan6868
silencoo
gmorso
Raphael-Stellwag
jiawanchun
davlasry
gnahtcouq
mcole-pl
faithinker
GetTKd
ttiee
HuangTM23
owllyi
wutiphong-sir
Boowx
XXXM1R0XXX
hromov
MajorTom3K1M
Kevinma999
iSpring
Alessandroinfo
zwzwoody
LeeFeee
hanbinz336-web
SPACEX-2022
such-stupid6
yuu19
Eastwindnovice
icodemo
cheikh-sadbouh
ehanc69
brusch
dm-ytlds
kesavan22
Zhang-ze-peng
shdwkl
palin83157701
wulangshu
hust-ropz
wansho
Eveosev
justpeterpan
Allenice
diandianxinchen
sky-goldfish
thantthet
yuebing-yb
L040610
star-io
WadeQi
sprince0031
Yjx710
gjf7
tianliangs
dean2021
fade03
pullp
prooval
tianqihou
DOCTZ3
Subikhyat1996
nikKrian
zhanghaonan777
ddg1024
isharebox
eizan97
jokermonn
enneket
rickkky
youjin-10
hyd396777507
se1987
Cyptopimpinainteazy
fezhang
MohammedSaudAlsahli
Charlsz
4everhope
mnbreno
dsvieira1990
underscoredevally
atthatmatt
rxqbi
mauvernaz
DanASMelo
kun1s2
jan-314
styluxlive
altar31
NakedoShadow
Liaming
404EVANDRO
XiaomingX
benjamibono
GrupoGlobalpb
pjyone
Tianket
hxj14
"""

# ===========================================

usernames = [u.strip() for u in usernames_input.strip().split('\n') if u.strip()]
print(f'Loaded {len(usernames)} usernames')
if GITHUB_COOKIE:
    print('GitHub cookie provided — will see profile emails')
else:
    print('No GitHub cookie — will only get emails from commit history')

In [ ]:
# Run the scraper
pw, browser, page = await start_browser(github_cookie=GITHUB_COOKIE if GITHUB_COOKIE else None)
results = []
found_count = 0

try:
    for username in tqdm(usernames, desc='Scraping emails'):
        email = await scrape_email(page, username)
        results.append({'username': username, 'email': email})

        if email:
            found_count += 1
            print(f'  ✓ {username} -> {email}')
        else:
            print(f'  ✗ {username} -> not found')

        await page.wait_for_timeout(1000)  # be polite
finally:
    await stop_browser(pw, browser)

print(f'\nDone! Found {found_count}/{len(usernames)} emails')

In [ ]:
# Results
df = pd.DataFrame(results)
print(f'Total: {len(df)}')
print(f'With email: {df["email"].notna().sum()}')
print(f'Without email: {df["email"].isna().sum()}')
print()

# Show all results
display(df)

# Save and download CSV
csv_filename = 'stargazer_emails.csv'
df.to_csv(csv_filename, index=False)

from google.colab import files
files.download(csv_filename)
print(f'\nDownloading {csv_filename}...')